In [94]:
# Libraries for data preparation & visualization
import numpy as np
import polars as pl
import pandas as pd

In [96]:
# Use your path in local
book   = pl.read_csv("./datalake/clean_merge/books.csv", schema={
    'isbn13': pl.String,
    'title': pl.String,
    'crawl_source': pl.String,
    'isbn10': pl.String,
    'description': pl.String,
    'image_url': pl.String,
    'author': pl.String,
    'language': pl.String,
    'publish_date': pl.String,
    'num_pages': pl.Int64
}, new_columns=[
    'isbn13','title','crawl_source','isbn10'
])

book_genre = pl.read_csv('./datalake/clean_merge/book_genres.csv', schema={
    'isbn13': pl.String,
    'genre_name': pl.String,
})

In [97]:
print(book)
print(book_genre)

shape: (284_893, 10)
┌───────────────┬────────────────┬──────────────┬────────────┬───┬───────────────┬──────────┬──────────────┬───────────┐
│ isbn13        ┆ title          ┆ crawl_source ┆ isbn10     ┆ … ┆ author        ┆ language ┆ publish_date ┆ num_pages │
│ ---           ┆ ---            ┆ ---          ┆ ---        ┆   ┆ ---           ┆ ---      ┆ ---          ┆ ---       │
│ str           ┆ str            ┆ str          ┆ str        ┆   ┆ str           ┆ str      ┆ str          ┆ i64       │
╞═══════════════╪════════════════╪══════════════╪════════════╪═══╪═══════════════╪══════════╪══════════════╪═══════════╡
│ 9783946642879 ┆ Hattest du     ┆ goodreads    ┆ 394664287X ┆ … ┆ Peer          ┆ German   ┆ 2020-04-01   ┆ 36        │
│               ┆ eigentlich     ┆              ┆            ┆   ┆ Jongeling     ┆          ┆              ┆           │
│               ┆ schon di…      ┆              ┆            ┆   ┆               ┆          ┆              ┆           │
│ 978236852

In [98]:
book_genre_joined = (
    book_genre
    .group_by("isbn13")
    .agg(
        pl.col("genre_name").str.concat(" ").alias("joined_genres"),
        pl.count("genre_name").alias("genre_count")
    )
)
print(book_genre_joined.describe())
book_genre_joined.sort('genre_count', descending=True)

shape: (9, 4)
┌────────────┬───────────────┬─────────────────────────────────┬─────────────┐
│ statistic  ┆ isbn13        ┆ joined_genres                   ┆ genre_count │
│ ---        ┆ ---           ┆ ---                             ┆ ---         │
│ str        ┆ str           ┆ str                             ┆ f64         │
╞════════════╪═══════════════╪═════════════════════════════════╪═════════════╡
│ count      ┆ 197848        ┆ 197848                          ┆ 197848.0    │
│ null_count ┆ 0             ┆ 0                               ┆ 0.0         │
│ mean       ┆ null          ┆ null                            ┆ 3.448213    │
│ std        ┆ null          ┆ null                            ┆ 3.235914    │
│ min        ┆ 0-415-33024-6 ┆ -- 1872-1928 -- Juvenile liter… ┆ 1.0         │
│ 25%        ┆ null          ┆ null                            ┆ 1.0         │
│ 50%        ┆ null          ┆ null                            ┆ 2.0         │
│ 75%        ┆ null          ┆ null   

isbn13,joined_genres,genre_count
str,str,u32
"""9780142437179""","""Young Adult Fiction FICTION Hi…",50
"""9789022531921""","""Middle Earth (Imaginary place)…",46
"""9780199541898""","""Classical fiction Resistance t…",44
"""9780140862539""","""Comics & Graphic Novels High i…",41
"""9780393926033""","""Young adult Classics Travel Li…",39
…,…,…
"""9781599795362""","""Religion""",1
"""9783764384845""","""Architecture""",1
"""9781591450931""","""Devotional calendars""",1


In [99]:
book_joined = book.join(book_genre_joined, on='isbn13', how='left')
book_joined

isbn13,title,crawl_source,isbn10,description,image_url,author,language,publish_date,num_pages,joined_genres,genre_count
str,str,str,str,str,str,str,str,str,i64,str,u32
"""9783946642879""","""Hattest du eigentlich schon di…","""goodreads""","""394664287X""","""Peer Jongelings Debutcomic füh…","""https://images-na.ssl-images-a…","""Peer Jongeling""","""German""","""2020-04-01""",36,"""LGBT Short Stories Transgender…",9
"""9782368529539""","""My Home Hero - tome 10""","""hardcover""","""2368529535""",null,null,"""Masashi Asaki""",null,null,0,"""Comics""",1
"""9780590439671""","""The Story Of Ruby Bridges""","""hardcover""","""0590439677""","""Summary, For months six-year-o…","""https://assets.hardcover.app/e…","""Robert Coles""","""English""","""1993-01-01""",32,"""African Americans Juvenile Non…",5
"""9780063216051""","""The Do-Over""","""hardcover""",null,"""From the author of the ""genuin…","""https://assets.hardcover.app/e…","""Suzanne Park""","""English""","""2023-01-01""",368,null,null
"""9798329760972""","""Wyvern's Gold""","""hardcover""","""1956455752""","""Six hundred years ago, humans …","""https://assets.hardcover.app/e…","""A.H. Hadley""",null,"""2024-07-26""",572,"""Dystopian""",1
…,…,…,…,…,…,…,…,…,…,…,…
"""9781683228615""","""Worry Less, Pray More: A Woman…","""hardcover""","""1683228618""",null,"""https://assets.hardcover.app/e…","""Donna K. Maltese""",null,null,0,null,null
"""9781250238016""","""The Copywriter's Handbook""","""hardcover""","""1250238013""","""The classic guide to copywriti…","""https://assets.hardcover.app/e…","""Robert W. Bly""","""English""","""1985-01-01""",432,"""Business & Economics""",1
"""9798322657149""","""The Pact""","""goodreads""","""None""","""𝐁𝐞𝐡𝐢𝐧𝐝 𝐇𝐞𝐫 𝐋𝐞𝐧𝐬The journey has…","""https://images-na.ssl-images-a…","""Helena Sage""","""English""","""2024-05-02""",501,"""Dark Reverse Harem Contemporar…",3


- EDA: Quality of each source:

In [100]:
book_joined_no_null = book_joined.filter(pl.col('joined_genres').is_null() == False)
book_joined_all_nulls = book_joined.filter(pl.col('joined_genres').is_null())

In [101]:
count_with_genres = book_joined_no_null['crawl_source'].value_counts().rename({
    'count':'Has genres'
})
count_without_genres = book_joined_all_nulls['crawl_source'].value_counts().rename({
    'count':'No genres'
})
count_with_genres.join(count_without_genres, on='crawl_source')

crawl_source,Has genres,No genres
str,u32,u32
"""goodreads""",43225,28516
"""hardcover""",154623,58529


In [55]:
book_joined_no_null

isbn13,title,crawl_source,isbn10,description,image_url,author,language,publish_date,num_pages,joined_genres,genre_count
str,str,str,str,str,str,str,str,str,i64,str,u32
"""9781438407166""","""Collaborative Construction of …","""hardcover""","""1438407165""","""The Collaborative Construction…","""https://assets.hardcover.app/e…","""Carollee Howes""","""English""","""1992-01-01""",172,"""Education""",1
"""9780751503890""","""The Physician""","""goodreads""","""0751503894""","""In the 11th century, Rob Cole …","""https://images-na.ssl-images-a…","""Noah Gordon""","""English""","""2001-07-01""",714,"""Classics Medicine Adventure No…",10
"""9783834915528""","""Operationalisierung einer Nach…","""hardcover""","""3834915521""","""Alexandro Kleine entwickelt ei…","""https://assets.hardcover.app/e…","""Alexandro Kleine""","""English""","""2009-02-26""",232,"""Business & Economics""",1
"""9782226109552""","""Lorsque j'étais une œuvre d'ar…","""hardcover""","""2226109552""","""Lorsque j'étais une œuvre d'ar…","""https://assets.hardcover.app/e…","""Éric-Emmanuel Schmitt""","""English""","""2002-01-01""",294,"""Fiction Classics""",2
"""9781421590851""","""Neon Genesis Evangelion 2-in-1…","""hardcover""","""1421590859""",null,"""https://assets.hardcover.app/e…","""Yoshiyuki Sadamoto""",null,"""2016-06-14""",0,"""Science Fiction Comics Young A…",6
…,…,…,…,…,…,…,…,…,…,…,…
"""9780593652930""","""The Queen of Dirt Island""","""hardcover""","""0593652932""","""“From its opening pages, this …","""https://assets.hardcover.app/e…","""Donal Ryan""","""English""","""2022-01-01""",257,"""Fiction""",1
"""9780515158212""","""What Is the World Cup?""","""hardcover""","""0515158216""","""GOOOAAAAAL! Get ready for a fr…","""https://assets.hardcover.app/e…","""Bonnie Bader""","""English""","""2018-01-01""",130,"""Juvenile Nonfiction""",1
"""9780316365819""","""The Witches: Salem, 1692""","""hardcover""","""0316200603""","""The Pulitzer Prize-winning aut…","""https://assets.hardcover.app/e…","""Stacy Schiff""","""English""","""2015-01-01""",498,"""History Women""",2


### Data Preprocessing

In [75]:
from unidecode import unidecode

In [69]:
book_joined_preprocessed = book_joined_no_null.with_columns(
    pl.col('joined_genres')
    .map_elements(unidecode, return_dtype=pl.String)  # Apply unidecode
    .str.to_lowercase()
    .str.replace_all(r'[^a-z0-9\s-]',' ')             # Remove non-alphanumeric
    .alias('fixed_genre')
)

- Check non-alphanumeric

In [74]:
test = book_joined_preprocessed.with_columns(
    pl.col("fixed_genre").str.replace_all(r'[a-zA-Z0-9 -]', '').alias("non_alphanumeric")
)

# test.filter(pl.col('fixed_genre').str.contains('qi'))
test.filter(pl.col('non_alphanumeric') != "").select(['isbn13','fixed_genre','non_alphanumeric'])['non_alphanumeric'].value_counts()

non_alphanumeric,count
str,u32


### Model implementation

In [77]:
book_df = book_joined_preprocessed.to_pandas()

In [76]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV

In [78]:
#removing stopwords
tfidf = TfidfVectorizer(stop_words='english')

#get the tf-idf scores
#create TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(book_df['fixed_genre'])

#shape of tfidf_matrix
tfidf_matrix.shape

(197714, 6081)

In [87]:
pd.DataFrame(tfidf_matrix[0].T.todense(), index=tfidf.get_feature_names_out(), columns=["tf-idf score"]).sort_values("tf-idf score")

,tf-idf score
000,0.0
measurement,0.0
meadow,0.0
md,0.0
mcnally,0.0
...,...
cyrano,0.0
cyprus,0.0
daisy,0.0
zukunft,0.0


In [90]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim1 = linear_kernel(tfidf_matrix, tfidf_matrix, dense_output=False)

MemoryError: Unable to allocate 82.2 GiB for an array with shape (11037802139,) and data type int64

In [91]:
# Use your path in local
book   = pl.read_csv("./datalake/clean_merge/books.csv", schema={
    'isbn13': pl.String,
    'title': pl.String,
    'crawl_source': pl.String,
    'isbn10': pl.String,
    'description': pl.String,
    'image_url': pl.String,
    'author': pl.String,
    'language': pl.String,
    'publish_date': pl.String,
    'num_pages': pl.Int64
}, new_columns=[
    'isbn13','title','crawl_source','isbn10'
])

user   = pl.read_csv("./datalake/clean_merge/users.csv", new_columns=['full_name','account_source','userid_source_original','userid'])
rating = pl.read_csv("./datalake/clean_merge/ratings.csv", schema_overrides={
    'isbn13': pl.String,
}, new_columns=[
    'rating_score','userid','isbn13','account_source'
])

print("Book:", book)
print("User:", user)
print("Rating:", rating)

Book: shape: (284_893, 10)
┌───────────────┬────────────────┬──────────────┬────────────┬───┬───────────────┬──────────┬──────────────┬───────────┐
│ isbn13        ┆ title          ┆ crawl_source ┆ isbn10     ┆ … ┆ author        ┆ language ┆ publish_date ┆ num_pages │
│ ---           ┆ ---            ┆ ---          ┆ ---        ┆   ┆ ---           ┆ ---      ┆ ---          ┆ ---       │
│ str           ┆ str            ┆ str          ┆ str        ┆   ┆ str           ┆ str      ┆ str          ┆ i64       │
╞═══════════════╪════════════════╪══════════════╪════════════╪═══╪═══════════════╪══════════╪══════════════╪═══════════╡
│ 9783946642879 ┆ Hattest du     ┆ goodreads    ┆ 394664287X ┆ … ┆ Peer          ┆ German   ┆ 2020-04-01   ┆ 36        │
│               ┆ eigentlich     ┆              ┆            ┆   ┆ Jongeling     ┆          ┆              ┆           │
│               ┆ schon di…      ┆              ┆            ┆   ┆               ┆          ┆              ┆           │
│ 978

In [93]:
# Count based on 'crawl_source', 'account_source' in books DataFrame
count_books = book.group_by('crawl_source').agg(pl.count().alias('book_count'))
count_users = user.group_by('account_source').agg(pl.count().alias('user_count'))
count_ratings = rating.group_by('account_source').agg(pl.count().alias('rating_count'))

# Show the results
print(count_books)
print(count_users)
print(count_ratings)

shape: (2, 2)
┌──────────────┬────────────┐
│ crawl_source ┆ book_count │
│ ---          ┆ ---        │
│ str          ┆ u32        │
╞══════════════╪════════════╡
│ hardcover    ┆ 213152     │
│ goodreads    ┆ 71741      │
└──────────────┴────────────┘
shape: (2, 2)
┌────────────────┬────────────┐
│ account_source ┆ user_count │
│ ---            ┆ ---        │
│ str            ┆ u32        │
╞════════════════╪════════════╡
│ goodreads      ┆ 23569      │
│ hardcover      ┆ 16745      │
└────────────────┴────────────┘
shape: (2, 2)
┌────────────────┬──────────────┐
│ account_source ┆ rating_count │
│ ---            ┆ ---          │
│ str            ┆ u32          │
╞════════════════╪══════════════╡
│ hardcover      ┆ 963997       │
│ goodreads      ┆ 445159       │
└────────────────┴──────────────┘


C:\Users\Legion\AppData\Local\Temp\ipykernel_11888\4114412882.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  count_books = book.group_by('crawl_source').agg(pl.count().alias('book_count'))
C:\Users\Legion\AppData\Local\Temp\ipykernel_11888\4114412882.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  count_users = user.group_by('account_source').agg(pl.count().alias('user_count'))
C:\Users\Legion\AppData\Local\Temp\ipykernel_11888\4114412882.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  count_ratings = rating.group_by('account_source').agg(pl.count().alias('rating_count'))


In [ ]:
cosine_sim

In [ ]:
def recommend(title, cosine_similarity=cosine_sim):
    index = programme_list.index(title)
    sim_score = list(enumerate(cosine_sim[index]))
    sim_score = sorted(sim_score, key= lambda x: x[1], reverse=True)[1:11] #position 0 is the movie itself, thus exclude
    recommend_index = [i[0] for i in sim_score]
    rec_movie = movie['title'].iloc[recommend_index]
    rec_score = [round(i[1],4) for i in sim_score]
    rec_table = pd.DataFrame(list(zip(rec_movie,rec_score)), columns=['Recommend programme','Similarity(0-1)'])
    return rec_table

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english'))  # Step 1: Apply TF-IDF
    
])

In [ ]:
# Fit and transform the data using the pipeline
grid_params = {
  'mnb__alpha': np.linspace(0.5, 1.5, 6),
  'mnb__fit_prior': [True, False],
  'tfidf_pip__tfidf_vectorizer__max_df': np.linspace(0.1, 1, 10),
  'tfidf_pip__tfidf_vectorizer__binary': [True, False],
  'tfidf_pip__tfidf_vectorizer__norm': [None, 'l1', 'l2'], 
}
clf = GridSearchCV(mnb_pipeline, grid_params)
clf.fit(train, y_train)
print("Best Score: ", clf.best_score_)
print("Best Params: ", clf.best_params_)


tfidf_pipeline.fit_transform(train)

tfidf_matrix = pipeline.fit_transform(df_pd['genre'])

# Convert the TF-IDF matrix to a DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=pipeline.named_steps['tfidf'].get_feature_names_out())

# Combine the original DataFrame with the TF-IDF values
df_pd = pd.concat([df_pd, tfidf_df], axis=1)